# Lab 2: Air Quality (PM2.5) Prediction (Regression)

**Task:** predict the PM2.5 concentration (fine particulate matter, the main ingredient of the air quality index) from other pollutant concentrations and weather measurements.

**Dataset:** Beijing Multi-Site Air Quality (UCI: https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data), included as `data/beijing_multi_site_air_quality_data.zip`, exactly as downloaded: hourly readings from 12 monitoring stations, March 2013 to February 2017.

**What we will do:**
1. Load the data and pick one station.
2. Clean the time series: build proper timestamps, treat missing values, deal with gaps.
3. Build a supervised table: PM2.5 at hour $t$ from pollutants and weather at hour $t$, plus PM2.5 at hours $t-1$ and $t-2$ (lag features).
4. Fit multiple linear regression, Ridge and Lasso, and look at the effect of dropping weakly correlated features.
5. Evaluate with $R^2$ and the distribution of errors; plot predicted against actual.
6. Report which pollutants matter most.

In [ ]:
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 25)

## 1. Loading the data

The download is a zip that contains *another* zip with one CSV per station (plus a few unrelated files). We open the inner zip directly from memory.

In [ ]:
outer = zipfile.ZipFile("data/beijing_multi_site_air_quality_data.zip")
print(outer.namelist())
inner = zipfile.ZipFile(io.BytesIO(outer.read("PRSA2017_Data_20130301-20170228.zip")))
station_files = [n for n in inner.namelist() if n.endswith(".csv")]
import re
station_name = lambda f: re.search(r"PRSA_Data_([A-Za-z]+)_\d", f).group(1)
print([station_name(f) for f in station_files])

In [ ]:
station = "Dongsi"                      # a station in central Beijing; try others later
raw = pd.read_csv(inner.open([f for f in station_files if f"_{station}_" in f][0]))
print(raw.shape)
raw.head()

Columns: `PM2.5`, `PM10`, `SO2`, `NO2`, `CO`, `O3` are pollutant concentrations (µg/m³); `TEMP` (°C), `PRES` (hPa), `DEWP` (dew point, °C), `RAIN` (mm), `wd` (wind direction, 16 compass points) and `WSPM` (wind speed, m/s) are weather. Date and time are spread over four columns.

## 2. Cleaning the time series

### Proper timestamps
Combine year, month, day and hour into one datetime, make it the index, and check that there is exactly one row per hour (a time series needs regular steps for "the previous hour" to mean anything).

In [ ]:
df = raw.copy()
df["time"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.set_index("time").sort_index().drop(columns=["No", "year", "month", "day", "hour", "station"])
print("rows:", len(df), " expected hourly rows:", len(pd.date_range(df.index.min(), df.index.max(), freq="h")))
print("duplicate timestamps:", df.index.duplicated().sum())

No hours are missing as rows. Missing *readings* are stored as empty values ("NA" in the file), which pandas already reads as NaN. Count them per column:

In [ ]:
missing = df.isna().sum()
print(pd.DataFrame({"missing": missing, "percent": (100 * missing / len(df)).round(2)}))

Missing values come in **runs** (a sensor out for a few hours or days), not as isolated values. How long are the gaps in PM2.5?

In [ ]:
def gap_lengths(series):
    is_na = series.isna()
    run_id = (is_na != is_na.shift()).cumsum()
    return is_na.groupby(run_id).sum()[lambda s: s > 0]

gaps = gap_lengths(df["PM2.5"])
print("number of gaps:", len(gaps), "| gaps of 1-3 hours:", (gaps <= 3).sum(), "| longer:", (gaps > 3).sum(), "| longest:", gaps.max(), "hours")
plt.figure(figsize=(10, 2.5))
plt.plot(df["PM2.5"].isna().astype(int).rolling(24 * 7).sum(), lw=0.8)
plt.ylabel("missing hours per week"); plt.title("PM2.5 missing readings over time"); plt.show()

### Treating the gaps

- **Short gaps (up to 3 hours):** pollution and weather change smoothly from hour to hour, so linear interpolation in time is a reasonable estimate. We fill these.
- **Long gaps:** interpolating across a day or two would invent data. We leave them missing, and any row that still has a missing value is dropped when we build the table.
- **Wind direction** is a category, so it can't be interpolated: we carry the last known direction forward for up to 3 hours.

In [ ]:
numeric_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
clean = df.copy()
clean[numeric_cols] = clean[numeric_cols].interpolate(method="time", limit=3, limit_area="inside")
clean["wd"] = clean["wd"].ffill(limit=3)
print("missing values left:", clean.isna().sum().sum(), "| rows with any missing value:", clean.isna().any(axis=1).sum())

## 3. Building the supervised table

Each row is one hour $t$. The **target** is PM2.5 at hour $t$. The **features** are:
- the other pollutants and the weather at the same hour $t$ (this is *estimating* PM2.5 from other measurements taken at the same time, for example when the PM2.5 sensor fails);
- **lag features**: PM2.5 at hours $t-1$ and $t-2$. Pollution builds up and clears over hours, so the recent past says a lot about now;
- the hour of the day and the month (traffic and heating follow daily and seasonal patterns);
- wind direction, one-hot encoded.

A lag is just the column shifted down: `shift(1)` puts the value from the previous row (the previous hour) in the current row. Because our index has exactly one row per hour, "previous row" really means "previous hour". The first rows have no history, and lags taken across a long gap are missing; those rows are dropped.

**Be careful with lags:** a feature must only contain information available at time $t$. PM2.5 at $t-1$ is fine; PM2.5 at $t$ itself, or a moving average that includes hour $t$, would be the answer hidden in a feature.

In [ ]:
table = clean.copy()
table["PM2.5_lag1"] = table["PM2.5"].shift(1)
table["PM2.5_lag2"] = table["PM2.5"].shift(2)
table["hour"] = table.index.hour
table["month"] = table.index.month
table = pd.get_dummies(table, columns=["wd"], prefix="wd", dtype=float)
table = table.dropna()
print(table.shape)
table[["PM2.5", "PM2.5_lag1", "PM2.5_lag2", "PM10", "CO", "TEMP"]].head()

### Splitting in time order

Neighbouring hours are almost identical. A random split would put 10 a.m. in training and 11 a.m. in testing, and the test score would mostly measure how well the model remembers yesterday's neighbours. In real use the model is applied to the future, so we test on the future: train on the first three years, test on the last.

In [ ]:
split = pd.Timestamp("2016-03-01")
train, test = table[table.index < split], table[table.index >= split]
target = "PM2.5"
pollutants = ["PM10", "SO2", "NO2", "CO", "O3"]
weather = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"] + [c for c in table.columns if c.startswith("wd_")]
lags = ["PM2.5_lag1", "PM2.5_lag2"]
calendar = ["hour", "month"]
print("train:", train.index.min().date(), "to", train.index.max().date(), len(train), "rows")
print("test :", test.index.min().date(), "to", test.index.max().date(), len(test), "rows")

`hour` and `month` are numbers, but 23:00 is next to 00:00 and December next to January, and their effect is not a straight line. We one-hot encode them for the linear models. All numeric features are standardised, which Ridge and Lasso need (their penalty treats all weights alike).

## 4. Models

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error

def make_model(features, regressor):
    num = [f for f in features if f not in calendar]
    cal = [f for f in features if f in calendar]
    prep = ColumnTransformer([("num", StandardScaler(), num),
                              ("cal", OneHotEncoder(handle_unknown="ignore"), cal)])
    return Pipeline([("prep", prep), ("model", regressor)])

results = []
def evaluate(name, features, regressor):
    model = make_model(features, regressor).fit(train[features], train[target])
    p_train, p_test = model.predict(train[features]), model.predict(test[features])
    results.append({"model": name, "features": len(features), "train R2": r2_score(train[target], p_train),
                    "test R2": r2_score(test[target], p_test), "test MAE": mean_absolute_error(test[target], p_test)})
    return model

all_features = pollutants + weather + calendar
evaluate("linear, same-hour features only", all_features, LinearRegression())
evaluate("linear, + lags t-1, t-2", all_features + lags, LinearRegression())
pd.DataFrame(results).round(3)

Same-hour pollutants and weather already explain most of PM2.5 ($R^2 \approx 0.9$); the lags add more. 

### Ridge and Lasso
The penalty strength is chosen by cross-validation *in time order* (`TimeSeriesSplit`: each fold trains on earlier data and validates on the period right after it), for the same reason we split by time.

In [ ]:
cv = TimeSeriesSplit(n_splits=5)
ridge = evaluate("ridge, + lags", all_features + lags, RidgeCV(alphas=np.logspace(-2, 3, 11), cv=cv))
lasso = evaluate("lasso, + lags", all_features + lags, LassoCV(alphas=np.logspace(-3, 1, 9), cv=cv, max_iter=20000))
print("ridge alpha:", ridge[-1].alpha_, " lasso alpha:", round(lasso[-1].alpha_, 4))
pd.DataFrame(results).round(3)

With 25,000 training hours and about 70 columns, the penalties change almost nothing: there's plenty of data for the number of weights, and Lasso picks a small alpha.

### Feature selection: dropping weakly correlated features
Which numeric features are only weakly correlated with PM2.5 in the training data?

In [ ]:
numeric_features = pollutants + ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"] + lags
corr = train[numeric_features + [target]].corr()[target].drop(target)
print(corr.round(3).sort_values(key=abs, ascending=False))
weak = corr[corr.abs() < 0.2].index.tolist()
print("weakly correlated (|r| < 0.2):", weak)

In [ ]:
selected = [f for f in all_features + lags if f not in weak and not f.startswith("wd_")]
evaluate("linear, + lags, weak features dropped", selected, LinearRegression())
pd.DataFrame(results).round(3)

Dropping the five weakly correlated features and wind direction leaves 9 features instead of 30, and the model is just as good (test $R^2$ 0.965 against 0.966, slightly lower MAE). Simpler, cheaper, easier to explain. But be careful with the conclusion "the weather doesn't matter": once PM2.5 an hour ago is known, most of the weather's effect is already in it. Without the lags, the weather matters a lot, as we'll see in section 6. And a weak correlation on its own doesn't prove a feature is useless: in a multiple regression a feature can matter *together with* others. Correlation filtering is a quick first look; always check the effect on validation or test performance, as we did here.

## 5. Evaluation

We use the plain linear model with lags (Ridge and Lasso were no better).

In [ ]:
final_features = all_features + lags
final = make_model(final_features, LinearRegression()).fit(train[final_features], train[target])
pred = pd.Series(final.predict(test[final_features]), index=test.index)
resid = test[target] - pred

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(resid, bins=100); axes[0].set_title("residuals (actual - predicted)")
axes[1].scatter(test[target], pred, s=2, alpha=0.3); axes[1].plot([0, 800], [0, 800], "k--")
axes[1].set_xlabel("actual PM2.5"); axes[1].set_ylabel("predicted PM2.5")
week = slice("2016-12-15", "2016-12-22")
axes[2].plot(test.loc[week, target], label="actual"); axes[2].plot(pred.loc[week], label="predicted")
axes[2].legend(); axes[2].set_title("one week in December 2016"); axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()
print(f"test R2 {r2_score(test[target], pred):.3f}, MAE {mean_absolute_error(test[target], pred):.1f} µg/m³")
print("residual percentiles 5/50/95:", np.percentile(resid, [5, 50, 95]).round(1))

Most errors are small, but the distribution has long tails: the largest errors happen during heavy pollution episodes (several hundred µg/m³), where the scatter widens. A linear model with a constant error spread can't capture that pollution episodes are both higher and more variable.

## 6. Which pollutants matter most?

The weights of a model with standardised features are "the change in PM2.5 for a one-standard-deviation change in the feature, holding the others fixed". The lags dominate a model that includes them, so to see the pollutants' own contribution we look at the model **without lags**:

In [ ]:
no_lag = make_model(all_features, LinearRegression()).fit(train[all_features], train[target])
names = no_lag[0].get_feature_names_out()
weights = pd.Series(no_lag[-1].coef_, index=[n.split("__")[1] for n in names])
print(weights[pollutants + ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]].round(2).sort_values(key=abs, ascending=False))

A second view: how much does the test $R^2$ fall when one pollutant is left out of the model?

In [ ]:
base = r2_score(test[target], no_lag.predict(test[all_features]))
drops = {}
for p in pollutants:
    feats = [f for f in all_features if f != p]
    m = make_model(feats, LinearRegression()).fit(train[feats], train[target])
    drops[p] = base - r2_score(test[target], m.predict(test[feats]))
print(pd.Series(drops).sort_values(ascending=False).round(4))

**Report.** PM10 (coarser particles, which include PM2.5 and come from the same sources) is by far the most important pollutant: leaving it out costs 0.14 in test $R^2$, everything else costs almost nothing. CO (combustion: traffic and coal heating) is second. Among the weather variables, dew point has a large positive weight (humid, stagnant air keeps fine particles suspended and helps them form). Wind speed is negatively correlated with PM2.5 on its own ($r = -0.30$, wind disperses pollution), but its weight is small once the pollutants are known: its effect already shows in them. NO2 and SO2 add little once PM10 and CO are known; they rise and fall with them. A caution about weights: correlated features (PM10, CO and NO2 all come from combustion) share credit in ways that can change from model to model, which is why the leave-one-out comparison is a useful second view.

### Optional: other stations
How well does the same recipe work at every station?

In [ ]:
def build_table(raw):
    """The same cleaning and features as above, as one function."""
    d = raw.copy()
    d["time"] = pd.to_datetime(d[["year", "month", "day", "hour"]])
    d = d.set_index("time").sort_index().drop(columns=["No", "year", "month", "day", "hour", "station"])
    d[numeric_cols] = d[numeric_cols].interpolate(method="time", limit=3, limit_area="inside")
    d["wd"] = d["wd"].ffill(limit=3)
    d["PM2.5_lag1"], d["PM2.5_lag2"] = d["PM2.5"].shift(1), d["PM2.5"].shift(2)
    d["hour"], d["month"] = d.index.hour, d.index.month
    return pd.get_dummies(d, columns=["wd"], prefix="wd", dtype=float).dropna()

rows = []
for f in station_files:
    d = build_table(pd.read_csv(inner.open(f)))
    tr, te = d[d.index < split], d[d.index >= split]
    feats = [c for c in final_features if c in d.columns]
    m = make_model(feats, LinearRegression()).fit(tr[feats], tr[target])
    rows.append({"station": station_name(f), "test R2": r2_score(te[target], m.predict(te[feats])),
                 "test MAE": mean_absolute_error(te[target], m.predict(te[feats]))})
pd.DataFrame(rows).sort_values("test R2").round(3)

The recipe works at every station ($R^2$ 0.96 to 0.97). A natural next step is to use the *other* stations' readings as features: pollution moves across the city with the wind.